In [1]:
!pip install transformers==4.41.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 1.0.0rc2
    Uninstalling huggingface-hub-1.0.0rc2:
      Successfully uninstalled huggingface-hub-1.0.0rc2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.2
    Uninstalling tokenizers-0.21.2:
      Successfully uninstalled tokenizers-0.21.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled transformers-4.53.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of

In [2]:
!pip install bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.3/321.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.8 MB/s eta 0:00:00


In [3]:
from transformers import T5Tokenizer, T5EncoderModel
import torch
import numpy as np
from tqdm.notebook import tqdm
from Bio import SeqIO
import gc
from pathlib import Path
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Available GPUs: {torch.cuda.device_count()}")

# Load ProtT5 model and tokenizer
tokenizer = T5Tokenizer.from_pretrained("Rostlab/prot_t5_xl_half_uniref50-enc", do_lower_case=False)
model = T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_half_uniref50-enc")

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

model = model.to(device)
model.eval()

def get_sequence_length_stats(sequences):
    lengths = [len(str(record.seq)) for record in sequences]
    return {
        'min': min(lengths),
        'max': max(lengths),
        'mean': np.mean(lengths),
        'median': np.median(lengths),
        'std': np.std(lengths)
    }

def prepare_sequence_for_t5(seq):
    """Add spaces between amino acids for T5 tokenization"""
    seq = str(seq).upper()
    # Replace rare/ambiguous amino acids
    seq = seq.replace('U', 'X').replace('Z', 'X').replace('O', 'X')
    # Add spaces between amino acids
    return ' '.join(list(seq))

def smart_batching(sequences, batch_size, max_length_diff=200):
    seq_data = [(i, record, len(str(record.seq))) 
                for i, record in enumerate(sequences)]
    seq_data.sort(key=lambda x: x[2])
    
    batches = []
    current_batch = []
    
    for idx, record, length in seq_data:
        if not current_batch:
            current_batch.append((idx, record, length))
        else:
            min_len = min(item[2] for item in current_batch)
            max_len = max(item[2] for item in current_batch)
            
            if (len(current_batch) < batch_size and 
                max(max_len, length) - min(min_len, length) <= max_length_diff):
                current_batch.append((idx, record, length))
            else:
                batches.append(current_batch)
                current_batch = [(idx, record, length)]
    
    if current_batch:
        batches.append(current_batch)
    
    return batches

def embed_fasta_prot_t5_optimized(fasta_path, output_prefix, batch_size=8, 
                                   max_seq_length=512, use_smart_batching=True,
                                   save_every=5000, use_fp16=True):
    
    if Path(f"{output_prefix}_embeddings.npy").exists():
        print(f"Embeddings already exist at {output_prefix}_embeddings.npy")
        return
    
    ids, embeds = [], []
    
    sequences = list(SeqIO.parse(fasta_path, "fasta"))
    total_seqs = len(sequences)
    print(f"Total sequences in {fasta_path}: {total_seqs}")
    
    stats = get_sequence_length_stats(sequences)
    print(f"\nSequence length statistics:")
    print(f"  Min: {stats['min']}, Max: {stats['max']}")
    print(f"  Mean: {stats['mean']:.1f}, Median: {stats['median']:.1f}")
    print(f"  Std: {stats['std']:.1f}")
    
    if use_smart_batching:
        print("\nCreating smart batches by sequence length...")
        batches = smart_batching(sequences, batch_size)
        print(f"Created {len(batches)} smart batches")
    else:
        batches = []
        for i in range(0, total_seqs, batch_size):
            batch_data = [(j, sequences[j], len(str(sequences[j].seq))) 
                         for j in range(i, min(i + batch_size, total_seqs))]
            batches.append(batch_data)
    
    result_embeds = [None] * total_seqs
    result_ids = [None] * total_seqs
    
    if use_fp16 and torch.cuda.is_available():
        print("Using mixed precision (FP16) for faster computation")
        model.half()
    
    processed = 0
    for batch_idx, batch_data in enumerate(tqdm(batches, desc="ProtT5 Embedding")):
        try:
            indices = [item[0] for item in batch_data]
            batch_records = [item[1] for item in batch_data]
            # Prepare sequences with spaces for T5
            batch_seqs = [prepare_sequence_for_t5(str(record.seq)[:max_seq_length]) 
                         for record in batch_records]
            batch_ids = [record.id for record in batch_records]
            
            # Tokenize sequences
            inputs = tokenizer.batch_encode_plus(
                batch_seqs,
                add_special_tokens=True,
                padding="longest",
                return_tensors='pt',
                truncation=True,
                max_length=max_seq_length
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            with torch.no_grad():
                if use_fp16 and torch.cuda.is_available():
                    with torch.cuda.amp.autocast():
                        outputs = model(**inputs)
                        embeddings = outputs.last_hidden_state
                else:
                    outputs = model(**inputs)
                    embeddings = outputs.last_hidden_state
                
                # Average pooling over sequence length
                attention_mask = inputs['attention_mask']
                mask_expanded = attention_mask.unsqueeze(-1).expand(embeddings.size()).float()
                sum_embeddings = torch.sum(embeddings * mask_expanded, dim=1)
                sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
                embeddings = (sum_embeddings / sum_mask).cpu().numpy()
                
                for idx, original_idx in enumerate(indices):
                    result_embeds[original_idx] = embeddings[idx]
                    result_ids[original_idx] = batch_ids[idx]
            
            processed += len(batch_data)
            
            if (processed % save_every == 0) or (batch_idx == len(batches) - 1):
                valid_embeds = [e for e in result_embeds if e is not None]
                valid_ids = [i for i in result_ids if i is not None]
                
                if valid_embeds:
                    np.save(f"{output_prefix}_embeddings_temp.npy", np.stack(valid_embeds))
                    np.save(f"{output_prefix}_ids_temp.npy", np.array(valid_ids))
                
                print(f"{processed}/{total_seqs} embeddings calculated | Checkpoint saved")
            
            if batch_idx % 100 == 0:
                torch.cuda.empty_cache()
                gc.collect()
                
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"\nOOM error at batch {batch_idx}. Clearing cache and retrying...")
                torch.cuda.empty_cache()
                gc.collect()
                continue
            else:
                raise e
    
    final_embeds = np.stack(result_embeds)
    final_ids = np.array(result_ids)
    
    np.save(f"{output_prefix}_embeddings.npy", final_embeds)
    np.save(f"{output_prefix}_ids.npy", final_ids)
    
    for temp_file in [f"{output_prefix}_embeddings_temp.npy", 
                      f"{output_prefix}_ids_temp.npy"]:
        if Path(temp_file).exists():
            Path(temp_file).unlink()
    
    print(f"\nFinished {output_prefix}: {len(final_embeds)} embeddings saved.")
    print(f"ProtT5 embeddings shape: {final_embeds.shape}")
    print(f"Embedding dimension: {final_embeds.shape[1]}")
    
    return final_embeds, final_ids

def embed_fasta_chunked(fasta_path, output_prefix, chunk_size=10000, **kwargs):
    sequences = list(SeqIO.parse(fasta_path, "fasta"))
    total_seqs = len(sequences)
    num_chunks = (total_seqs + chunk_size - 1) // chunk_size
    
    print(f"Processing {total_seqs} sequences in {num_chunks} chunks")
    
    all_embeds, all_ids = [], []
    
    for chunk_idx in range(num_chunks):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, total_seqs)
        
        print(f"\nProcessing chunk {chunk_idx + 1}/{num_chunks}")
        
        chunk_fasta = f"temp_chunk_{chunk_idx}.fasta"
        SeqIO.write(sequences[start_idx:end_idx], chunk_fasta, "fasta")
        
        chunk_embeds, chunk_ids = embed_fasta_prot_t5_optimized(
            chunk_fasta, 
            f"temp_chunk_{chunk_idx}",
            **kwargs
        )
        
        all_embeds.append(chunk_embeds)
        all_ids.extend(chunk_ids)
        
        Path(chunk_fasta).unlink()
        Path(f"temp_chunk_{chunk_idx}_embeddings.npy").unlink()
        Path(f"temp_chunk_{chunk_idx}_ids.npy").unlink()
    
    final_embeds = np.vstack(all_embeds)
    final_ids = np.array(all_ids)
    
    np.save(f"{output_prefix}_embeddings.npy", final_embeds)
    np.save(f"{output_prefix}_ids.npy", final_ids)
    
    print(f"\nAll chunks processed: {final_embeds.shape}")

# Process test sequences
embed_fasta_prot_t5_optimized(
    "/kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset.fasta", 
    "test", 
    batch_size=8,  # Reduced batch size for T5 (larger model)
    max_seq_length=512,  # T5 has shorter context
    use_smart_batching=True,
    use_fp16=True,
    save_every=5000
)

# Process train sequences
embed_fasta_prot_t5_optimized(
    "/kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta", 
    "train", 
    batch_size=8,
    max_seq_length=512,
    use_smart_batching=True,
    use_fp16=True,
    save_every=5000
)

Using device: cuda
Available GPUs: 2


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

/tmp/ipykernel_19/2495170747.py:131: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
2025-11-02 16:30:11.454857: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762101011.657576      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762101011.719404      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/tmp/ipykernel_19/2495170747.py:131: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


5000/224309 embeddings calculated | Checkpoint saved
10000/224309 embeddings calculated | Checkpoint saved
15000/224309 embeddings calculated | Checkpoint saved
20000/224309 embeddings calculated | Checkpoint saved
25000/224309 embeddings calculated | Checkpoint saved
30000/224309 embeddings calculated | Checkpoint saved
35000/224309 embeddings calculated | Checkpoint saved
40000/224309 embeddings calculated | Checkpoint saved
45000/224309 embeddings calculated | Checkpoint saved
50000/224309 embeddings calculated | Checkpoint saved
55000/224309 embeddings calculated | Checkpoint saved
60000/224309 embeddings calculated | Checkpoint saved
65000/224309 embeddings calculated | Checkpoint saved
70000/224309 embeddings calculated | Checkpoint saved
75000/224309 embeddings calculated | Checkpoint saved
80000/224309 embeddings calculated | Checkpoint saved
85000/224309 embeddings calculated | Checkpoint saved
90000/224309 embeddings calculated | Checkpoint saved
95000/224309 embeddings calcu

ProtT5 Embedding:   0%|          | 0/10313 [00:00<?, ?it/s]

5000/82404 embeddings calculated | Checkpoint saved
10000/82404 embeddings calculated | Checkpoint saved
15000/82404 embeddings calculated | Checkpoint saved
20000/82404 embeddings calculated | Checkpoint saved
25000/82404 embeddings calculated | Checkpoint saved
30000/82404 embeddings calculated | Checkpoint saved
35000/82404 embeddings calculated | Checkpoint saved
40000/82404 embeddings calculated | Checkpoint saved
45000/82404 embeddings calculated | Checkpoint saved
50000/82404 embeddings calculated | Checkpoint saved
55000/82404 embeddings calculated | Checkpoint saved
60000/82404 embeddings calculated | Checkpoint saved
65000/82404 embeddings calculated | Checkpoint saved
70000/82404 embeddings calculated | Checkpoint saved
75000/82404 embeddings calculated | Checkpoint saved
80000/82404 embeddings calculated | Checkpoint saved
82404/82404 embeddings calculated | Checkpoint saved

Finished train: 82404 embeddings saved.
ProtT5 embeddings shape: (82404, 1024)
Embedding dimension:

(array([[ 0.09927323,  0.04379587, -0.09390651, ...,  0.10315256,
         -0.02916089, -0.04594825],
        [ 0.02413786, -0.0234924 ,  0.03580002, ...,  0.02082824,
          0.04891568,  0.00628371],
        [ 0.02515895,  0.03666744,  0.07178067, ..., -0.03914816,
          0.0312604 ,  0.0466047 ],
        ...,
        [ 0.00949263,  0.12030818,  0.01663985, ...,  0.00836154,
         -0.01524515, -0.06031914],
        [ 0.0370216 ,  0.09100086,  0.00054257, ...,  0.06865664,
          0.02092885, -0.01077574],
        [ 0.0169507 ,  0.09628341,  0.00381133, ...,  0.10154507,
          0.01547833, -0.02660577]], dtype=float32),
 array(['sp|A0A0C5B5G6|MOTSC_HUMAN', 'sp|A0JNW5|BLT3B_HUMAN',
        'sp|A0JP26|POTB3_HUMAN', ..., 'sp|Q9Y7P7|YQ63_SCHPO',
        'sp|Q9Y7Q3|YQ6A_SCHPO', 'sp|Q9Y816|YOND_SCHPO'], dtype='<U25'))